#1.Word2Vec on IMDB Reviews
Training word embeddings on 50k movie reviews using Gensim's Word2Vec.

##1.1. Setup
### Install & import dependencies

In [1]:
! pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 69.1 MB/s eta 0:00:00


In [2]:
# !pip install gensim nltk pandas
import sys
import re
import logging
import warnings

import pandas as pd
import nltk
from nltk.tokenize import sent_tokenize
from gensim.models import Word2Vec
from gensim.utils import simple_preprocess

warnings.filterwarnings("ignore")
logging.basicConfig(format="%(asctime)s : %(levelname)s : %(message)s", level=logging.INFO)

nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)

True

##1.2. Load Dataset
### Read the IMDB CSV and do a quick sanity check

In [3]:
# from google.colab import drive
# drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
df = pd.read_csv('IMDB Dataset.csv')

df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [5]:
df.shape

(50000, 2)

In [6]:
df["sentiment"].value_counts()

,count
sentiment,
positive,25000
negative,25000


##1.3. Text Cleaning
### Strip HTML tags, keep only letters, lowercase

In [7]:
def clean_text(text):
    text = re.sub(r"<.*?>", " ", text)
    text = re.sub(r"[^a-zA-Z\s]", " ", text)
    return re.sub(r"\s+", " ", text).strip().lower()

df["cleaned"] = df["review"].apply(clean_text)

In [8]:
# spot check
print("Before:", df["review"].iloc[0])
print("After :", df["cleaned"].iloc[0])

Before: One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.<br /><br />The first thing that struck me about Oz was its brutality and unflinching scenes of violence, which set in right from the word GO. Trust me, this is not a show for the faint hearted or timid. This show pulls no punches with regards to drugs, sex or violence. Its is hardcore, in the classic use of the word.<br /><br />It is called OZ as that is the nickname given to the Oswald Maximum Security State Penitentary. It focuses mainly on Emerald City, an experimental section of the prison where all the cells have glass fronts and face inwards, so privacy is not high on the agenda. Em City is home to many..Aryans, Muslims, gangstas, Latinos, Christians, Italians, Irish and more....so scuffles, death stares, dodgy dealings and shady agreements are never far away.<br /><br />I would say the main appeal of the show is due to

##1.4. Sentence Tokenisation
### Split each review into individual sentences

In [10]:
sentences_raw = []
for review in df["cleaned"]:
    sentences_raw.extend(sent_tokenize(review))

In [11]:
print("After :", df["cleaned"].iloc[0])

After : one of the other reviewers has mentioned that after watching just oz episode you ll be hooked they are right as this is exactly what happened with me the first thing that struck me about oz was its brutality and unflinching scenes of violence which set in right from the word go trust me this is not a show for the faint hearted or timid this show pulls no punches with regards to drugs sex or violence its is hardcore in the classic use of the word it is called oz as that is the nickname given to the oswald maximum security state penitentary it focuses mainly on emerald city an experimental section of the prison where all the cells have glass fronts and face inwards so privacy is not high on the agenda em city is home to many aryans muslims gangstas latinos christians italians irish and more so scuffles death stares dodgy dealings and shady agreements are never far away i would say the main appeal of the show is due to the fact that it goes where other shows wouldn t dare forget p

In [12]:
len(sentences_raw)

50000

In [13]:
len(sentences_raw) / len(df)

1.0

In [14]:
print("  First 3 sentences:")
for s in sentences_raw[:3]:
    print('.',s)

  First 3 sentences:
. one of the other reviewers has mentioned that after watching just oz episode you ll be hooked they are right as this is exactly what happened with me the first thing that struck me about oz was its brutality and unflinching scenes of violence which set in right from the word go trust me this is not a show for the faint hearted or timid this show pulls no punches with regards to drugs sex or violence its is hardcore in the classic use of the word it is called oz as that is the nickname given to the oswald maximum security state penitentary it focuses mainly on emerald city an experimental section of the prison where all the cells have glass fronts and face inwards so privacy is not high on the agenda em city is home to many aryans muslims gangstas latinos christians italians irish and more so scuffles death stares dodgy dealings and shady agreements are never far away i would say the main appeal of the show is due to the fact that it goes where other shows wouldn 

##1.5. Tokenisation with Gensim
### Convert sentences to token lists using `simple_preprocess`

In [15]:
sentences = [
    simple_preprocess(s, deacc=True, min_len=2, max_len=15)
    for s in sentences_raw
    if simple_preprocess(s, deacc=True, min_len=2, max_len=15)
]

In [16]:
total_tokens = sum(len(s) for s in sentences)
print('Total sentences:',len(sentences))
print('Total tokens:', (total_tokens))

Total sentences: 50000
Total tokens: 10975386


In [17]:
print("First review tokens:")
for s in sentences[:1]:
    print(s[:])

First review tokens:
['one', 'of', 'the', 'other', 'reviewers', 'has', 'mentioned', 'that', 'after', 'watching', 'just', 'oz', 'episode', 'you', 'll', 'be', 'hooked', 'they', 'are', 'right', 'as', 'this', 'is', 'exactly', 'what', 'happened', 'with', 'me', 'the', 'first', 'thing', 'that', 'struck', 'me', 'about', 'oz', 'was', 'its', 'brutality', 'and', 'unflinching', 'scenes', 'of', 'violence', 'which', 'set', 'in', 'right', 'from', 'the', 'word', 'go', 'trust', 'me', 'this', 'is', 'not', 'show', 'for', 'the', 'faint', 'hearted', 'or', 'timid', 'this', 'show', 'pulls', 'no', 'punches', 'with', 'regards', 'to', 'drugs', 'sex', 'or', 'violence', 'its', 'is', 'hardcore', 'in', 'the', 'classic', 'use', 'of', 'the', 'word', 'it', 'is', 'called', 'oz', 'as', 'that', 'is', 'the', 'nickname', 'given', 'to', 'the', 'oswald', 'maximum', 'security', 'state', 'penitentary', 'it', 'focuses', 'mainly', 'on', 'emerald', 'city', 'an', 'experimental', 'section', 'of', 'the', 'prison', 'where', 'all', 't

##1.6. Build & Train Word2Vec
### Initialise the model, build vocab, then train

In [19]:
# remove all existing handlers first
logging.root.handlers = []

# basicConfig will apply
logging.basicConfig(
    stream=sys.stdout,
    format="%(asctime)s : %(levelname)s : %(message)s",
    level=logging.INFO
)

In [20]:
model = Word2Vec(
    vector_size=100,
    window=5,
    min_count=2,
    workers=4,
    # To apply Skip-Gram, if use 0 then cbow
    sg=1,
    # no of iteration for neural network learning
    epochs=5
)

model.build_vocab(sentences, progress_per=5_000)
print(f"Vocab size : {len(model.wv):,} words")

model.train(sentences, total_examples=model.corpus_count, epochs=model.epochs)
print("Training done.")

2026-07-15 15:26:45,691 : INFO : Word2Vec lifecycle event {'params': 'Word2Vec<vocab=0, vector_size=100, alpha=0.025>', 'datetime': '2026-07-15T15:26:45.691378', 'gensim': '4.4.0', 'python': '3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]', 'platform': 'Linux-6.6.122+-x86_64-with-glibc2.35', 'event': 'created'}
2026-07-15 15:26:45,692 : INFO : collecting all words and their counts
2026-07-15 15:26:45,693 : INFO : PROGRESS: at sentence #0, processed 0 words, keeping 0 word types
2026-07-15 15:26:45,856 : INFO : PROGRESS: at sentence #5000, processed 1097850 words, keeping 38380 word types
2026-07-15 15:26:46,053 : INFO : PROGRESS: at sentence #10000, processed 2194344 words, keeping 51598 word types
2026-07-15 15:26:46,256 : INFO : PROGRESS: at sentence #15000, processed 3299308 words, keeping 60852 word types
2026-07-15 15:26:46,454 : INFO : PROGRESS: at sentence #20000, processed 4390112 words, keeping 68284 word types
2026-07-15 15:26:46,663 : INFO : PROGRESS: at sentence #25000,

##1.7. Save Model
### Persist the trained model to disk

In [21]:
model.save("word2vec_imdb.model")
print("Saved → word2vec_imdb.model")

2026-07-15 15:35:20,936 : INFO : Word2Vec lifecycle event {'fname_or_handle': 'word2vec_imdb.model', 'separately': 'None', 'sep_limit': 10485760, 'ignore': frozenset(), 'datetime': '2026-07-15T15:35:20.936351', 'gensim': '4.4.0', 'python': '3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]', 'platform': 'Linux-6.6.122+-x86_64-with-glibc2.35', 'event': 'saving'}
2026-07-15 15:35:20,937 : INFO : not storing attribute cum_table
2026-07-15 15:35:21,009 : INFO : saved word2vec_imdb.model
Saved → word2vec_imdb.model


##1.8. Explore Embeddings
### Similar words, analogies, and cosine similarities

In [ ]:
# most similar words
for word in ["good", "terrible", "actor", "film", "love"]:
    if word in model.wv:
        similar = model.wv.most_similar(word, topn=5)
        print(f"\n{word}:")
        for w, score in similar:
            print(f"  {w:<20} {score:.4f}")

In [ ]:
# word vector peek
print(model.wv["movie"])

In [ ]:
model.wv["movie"].shape

In [ ]:
# analogy
result = model.wv.most_similar(positive=["good", "terrible"], negative=["bad"], topn=3)
print("good - bad + terrible ≈")
for w, score in result:
    print(f"  {w:<20} {score:.4f}")

In [ ]:
# pairwise cosine similarity
pairs = [("good", "great"), ("good", "worse"), ("movie", "film")]
for w1, w2 in pairs:
    sim = model.wv.similarity(w1, w2)
    print(f"{w1} ↔ {w2:<10} {sim:.4f}")

In [ ]:
model.wv.doesnt_match(['actor', 'director', 'screenplay', 'popcorn'])

In [ ]:
model.wv.doesnt_match(['horror', 'thriller', 'suspense', 'romance'])

- **How its working?**
1. Get vector for each word
2. Compute the mean vector of the whole group
3. Find which word's vector is most distant from that mean
4. Return that word

##1.9. VIsualizaing 3d vector space

In [ ]:
y = model.wv.index_to_key

from sklearn.decomposition import PCA
pca = PCA(n_components=3)
X = pca.fit_transform(model.wv.get_normed_vectors())

In [ ]:
import plotly.express as px
fig = px.scatter_3d(X[200:300],x=0,y=1,z=2, color=y[200:300])
fig.show()



---



---



#2.Sentimental Analysis Model Building

##2.1.Embedding tfidf with vectors space

In [ ]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer

In [ ]:
# fit tfidf to get word weights (don't use matrix, just the vocab + idf scores)
tfidf = TfidfVectorizer(max_features=80000)
tfidf.fit(df["cleaned"])

word2weight = dict(zip(tfidf.get_feature_names_out(), tfidf.idf_))

In [ ]:
def review_vector(review):
    tokens = simple_preprocess(review, deacc=True, min_len=2, max_len=15)
    vectors = [
        model.wv[w] * word2weight.get(w, 1.0)
        for w in tokens if w in model.wv
    ]
    return np.mean(vectors, axis=0) if vectors else np.zeros(model.vector_size)

- Word2Vec builds the semantic space. TF-IDF decides how much of each word's meaning to inject into the final review vector.



In [ ]:
X = np.vstack(df["cleaned"].apply(review_vector))
y = (df["sentiment"] == "positive").astype(int)

print(f"Feature matrix : {X.shape}")
print(f"Label balance  : {y.value_counts().to_dict()}")

In [ ]:
X[0].shape

##2.2.Data splitting

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train : {X_train.shape[0]:,}  |  Test : {X_test.shape[0]:,}")

##2.3.Model building using Logistic Regression

In [ ]:
from sklearn.linear_model import LogisticRegression

clf = LogisticRegression(max_iter=1000, C=1.0, solver="lbfgs", n_jobs=-1)
clf.fit(X_train, y_train)
print("Model trained.")

##2.4.Evaluation using classification report

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

y_pred = clf.predict(X_test)

display(classification_report(y_test, y_pred, target_names=["negative", "positive"]))

cm = confusion_matrix(y_test, y_pred)
ConfusionMatrixDisplay(cm, display_labels=["negative", "positive"]).plot(cmap="Blues")
plt.title("Confusion Matrix")
plt.show()

In [ ]:
def predict(review):
    vec = review_vector(clean_text(review)).reshape(1, -1)
    pred = clf.predict(vec)[0]
    prob = clf.predict_proba(vec)[0]
    label = "positive" if pred == 1 else "negative"
    print(f"Review   : {review[:80]}")
    print(f"Sentiment: {label}  (confidence: {max(prob):.2%})")

In [ ]:
predict("The movie had great performances but the plot was absolutely terrible.")

In [ ]:
predict("One of the best films I've seen this year. Brilliantly directed.")